# **importing the needed libraries**

In [ ]:
!pip install numpy==1.25.2

In [ ]:
!pip install gensim

In [ ]:
pip install fasttext

In [ ]:
import re
import nltk
import string
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem.isri import ISRIStemmer
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models import Word2Vec
from gensim.models import FastText
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
import fasttext
from sklearn.model_selection import train_test_split

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
!pip install camel-tools[full]

# **importing the dataset**

In [ ]:
patient_dr = pd.read_csv("/content/altibbi_specialty_data.csv")

In [ ]:
patient_dr.head()

In [ ]:
#the data is big so we took 25000 random samples of it
df = patient_dr.drop("name_ar", axis=1)
df = df.sample(n=25000, random_state=42)

In [ ]:
df.head()

# **Pre-processing**

***Pre-processing using Iris stemming (NLTK)***

In [ ]:
def preprocessing_patient_stemming(patient_Question):
    # setting a standard
    patient_Question = re.sub(r'[إأآ]', 'ا', patient_Question)
    patient_Question = re.sub(r'ؤ', 'و', patient_Question)
    patient_Question = re.sub(r'ئ', 'ي', patient_Question)
    patient_Question = re.sub(r'ء', '', patient_Question)
    patient_Question = re.sub(r'ة', 'ه', patient_Question)
    # removing harkat
    patient_Question = re.sub(r'[\u064B-\u0652]', '', patient_Question)
    # Removing punctuation
    patient_Question = re.sub(r'[^\w\s]', ' ', patient_Question)
    # tokenization
    tokens = nltk.word_tokenize(patient_Question)
    # Remove stopwords
    ar_stopwords = set(stopwords.words('arabic'))
    without_stop = [word for word in tokens if word not in ar_stopwords]
    # stemming
    stemmer = ISRIStemmer()
    steammed = [stemmer.stem(word) for word in without_stop]
    return steammed

In [ ]:
preprocessed_Question_Stemming = [preprocessing_patient_stemming(text) for text in df['question_body']]

In [ ]:
for question, processed_qu in list(zip(df['question_body'], preprocessed_Question_Stemming))[:100]:
    print(f" Question: {question}")
    print(f" Processed Question: {processed_qu}\n")

***Pre-processing using porter stemmer***

In [ ]:
!camel_data -i light

In [ ]:
from camel_tools.disambig.mle import MLEDisambiguator
disambig = MLEDisambiguator.pretrained()
def preprocessing_patient_camel(patient_Question):
    # Removing punctuation
    patient_Question = re.sub(r'[^\w\s]', ' ', patient_Question)
    #only returning the lexemma of the word using camel tools morphological analysis(lemmatization)
    analysis_output = disambig.disambiguate(patient_Question.split())
    lexemma = [res.analyses[0].analysis.get('lex', res.word) for res in analysis_output if res.analyses]
    ar_stopwords = set(stopwords.words('arabic'))
    ready= []
    for word in lexemma:
      # setting a standard
      word = re.sub(r'[إأآ]', 'ا', word)
      word = re.sub(r'ؤ', 'و', word)
      word = re.sub(r'ئ', 'ي', word)
      word = re.sub(r'ء', '', word)
      word = re.sub(r'ة', 'ه', word)
      # removing harkat
      word = re.sub(r'[\u064B-\u0652]', '', word)
      if word not in ar_stopwords:
        ready.append(word)
    return ready


In [ ]:
preprocessed_Question_camel = [preprocessing_patient_camel(text) for text in df['question_body']]

In [ ]:
for question, processed_qu in list(zip(df['question_body'], preprocessed_Question_camel))[:100]:
    print(f" Question: {question}")
    print(f" Processed Question: {processed_qu}\n")

***Pre-processing using Simple Lemmetation (RE)***

In [ ]:
def preprocessing_patient_RE_lemmatization (patient_Question):
    # setting a standard
    patient_Question = re.sub(r'[إأآ]', 'ا', patient_Question)
    patient_Question = re.sub(r'ؤ', 'و', patient_Question)
    patient_Question = re.sub(r'ئ', 'ي', patient_Question)
    patient_Question = re.sub(r'ء', '', patient_Question)
    patient_Question = re.sub(r'ة', 'ه', patient_Question)
    # removing harkat
    patient_Question = re.sub(r'[\u064B-\u0652]', '', patient_Question)
    # Removing punctuation
    patient_Question = re.sub(r'[^\w\s]', ' ', patient_Question)
    # tokenization
    tokens = nltk.word_tokenize(patient_Question)
    # Remove stopwords
    ar_stopwords = set(stopwords.words('arabic'))
    without_stop = [word for word in tokens if word not in ar_stopwords]
    #lemmatization using regular ewxpressions where we manualy remove the parts of the word that are not from the lexemma
    lemmatized_re = [re.sub(r'(هات|ات|ون|ين|ان|يه|ها|هم|كم|نا|ه|ي|ة|ا)$', '', word) for word in without_stop]
    return lemmatized_re

In [ ]:
preprocessed_Question_RE_lemmatized = [preprocessing_patient_RE_lemmatization(text) for text in df['question_body']]

In [ ]:
for question, processed_qu in list(zip(df['question_body'], preprocessed_Question_RE_lemmatized))[:100]:
  print(f" Question: {question}")
  print(f" Processed Question: {processed_qu}\n")

best preprocessing was using camel tools

In [ ]:
#saving the data to use it neural models notebook
data_dic = {
    'x': [' '.join(tokens) for tokens in preprocessed_Question_camel],
    'y': df['specialty_id'].values}
preprocessed_df = pd.DataFrame(data_dic)


In [ ]:
preprocessed_df.shape

In [ ]:
preprocessed_df.to_csv('ready_data.csv', index=False)

#**testing different word representation on logistic regression to know which word representation we should use for the rest**

**Traditional word representation:**

First we have to join the text because the input should be joined i choose camel tools for preprocessing

In [ ]:
join_questions_camel = [' '.join(word) for word in preprocessed_Question_camel]

**BOW Bag of words**

In [ ]:
BOW = CountVectorizer()

***BOW - Bag Of Word with  (camel)***

In [ ]:
BOW_representation = BOW.fit_transform(join_questions_camel)

***splitting***

In [ ]:
X_train_BOW_camel, X_test_BOW_camel, y_train_BOW_camel, y_test_BOW_camel = train_test_split(BOW_representation,df['specialty_id'][:BOW_representation.shape[0]], test_size=0.2 , random_state=42)

**TF_IDF**

In [ ]:
TF_IDF = TfidfVectorizer()

In [ ]:
TF_IDF_representation = TF_IDF.fit_transform(join_questions_camel)

***splitting***

In [ ]:
X_train_TFIDF_camel, X_test_TFIDF_camel, y_train_TFIDF_camel, y_test_TFIDF_camel = train_test_split(TF_IDF_representation, df['specialty_id'][:TF_IDF_representation.shape[0]], test_size=0.2 , random_state=42)

***Logistic Regression - BOW***

In [ ]:
LR_BOW = LogisticRegression()
LR_BOW.fit(X_train_BOW_camel.toarray(), y_train_BOW_camel)
y_pred_LR_BOW = LR_BOW.predict(X_test_BOW_camel.toarray())
print(classification_report(y_test_BOW_camel, y_pred_LR_BOW))

***Logistic Regression - TFIDF***

In [ ]:
LR_TFIDF= LogisticRegression()
LR_TFIDF.fit(X_train_TFIDF_camel.toarray(), y_train_TFIDF_camel)
y_pred_LR_TFIDF = LR_TFIDF.predict(X_test_TFIDF_camel.toarray())
print(classification_report(y_test_TFIDF_camel, y_pred_LR_TFIDF))

**Non-Traditional word representation**

In [ ]:
#unlike traditional word representation, the model doesnt return a vector to each sentance
#thus we need to take the frequancy of each world by searching it in the model, then take its average to form a vector for each sentance
def average_vec (model, sentance_tokens):
  word_representation = []
  for word in sentance_tokens:
    if word in model.wv:
      word_representation.append(model.wv[word])
    else: #if the word is not found appeand a vec of zeros
      word_representation.append(np.zeros(150))
    #calculating the average
    return np.mean(word_representation, axis=0)

***Word2vec with camel***

In [ ]:
!wget https://archive.org/download/full_grams_cbow_300_twitter/full_uni_cbow_100_twitter.zip
!unzip full_uni_cbow_100_twitter.zip

In [ ]:
from gensim.models import Word2Vec
Word2vec= Word2Vec.load("full_uni_cbow_100_twitter.mdl")

In [ ]:
w2v_embeddings =[]
for x in preprocessed_Question_camel:
  v = average_vec(Word2vec, x)
  if v is not None and v.shape == (Word2vec.vector_size,):
    w2v_embeddings.append(v)

word2vec_representation = np.vstack(w2v_embeddings)

In [ ]:
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(word2vec_representation, df['specialty_id'][:len(word2vec_representation)], test_size=0.2 , random_state=42)

***fasttext with camel***

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz

In [ ]:
!gunzip cc.ar.300.bin.gz

In [ ]:
#loading the already pretrained fasttext
Fasttext = fasttext.load_model('cc.ar.300.bin')

In [ ]:
#step 1: unlike traditional word representation, the model doesnt return a vector to each sentance
#thus we need to take the frequancy of each world by searching it in the model, then take its average to form a vector for each sentance

def average_vec_ft(model, sentence_tokens):
    word_representation = []
    for word in sentence_tokens:
        try:
            word_representation.append(model.get_word_vector(word))
        except: #if the word is not found appeand a vec of zeros
            word_representation.append(np.zeros(model.get_dimension()))
    #calculating the average
    return np.mean(word_representation, axis=0) if word_representation else np.zeros(model.get_dimension())

In [ ]:
#averaging the vectors of fasttext
ft_embeddings =[]
for x in preprocessed_Question_camel:
  v = average_vec_ft(Fasttext, x)
  if v is not None and v.shape == (Fasttext.get_dimension(),):
    ft_embeddings.append(v)

In [ ]:
#making sure that only the labels of the valid fasttext embeddings are saved
ft_labels = []
for x,w in enumerate(preprocessed_Question_camel):
  v = average_vec_ft(Fasttext, w)
  if v is not None and v.shape == (Fasttext.get_dimension(),):
    ft_labels.append(df['specialty_id'].iloc[x])

In [ ]:
X_train_ft, X_test_ft, y_train_ft, y_test_ft = train_test_split( np.vstack(ft_embeddings), ft_labels, test_size=0.2, random_state=42)

**Logistic Regression - word2vec**

In [ ]:
LR_w2v = LogisticRegression(max_iter = 1000)
LR_w2v.fit(X_train_w2v, y_train_w2v)
y_pred_LR_w2v = LR_w2v.predict(X_test_w2v)
print(classification_report(y_test_w2v, y_pred_LR_w2v))

**Logistic Regression- FastText**

In [ ]:
LR_FT = LogisticRegression(max_iter = 1000)
LR_FT.fit(X_train_ft, y_train_ft)
y_pred_LR_FT = LR_FT.predict(X_test_ft)
print(classification_report(y_test_ft, y_pred_LR_FT, zero_division=0))

# *Best results were using tf-idf*

# **Traditional machine learning models classification**

***Naive Bayes - camel - TFIDF***

In [ ]:
NB_TFIDF = GaussianNB()
NB_TFIDF.fit(X_train_TFIDF_camel.toarray(), y_train_TFIDF_camel)
y_pred_NB_TFIDF = NB_TFIDF.predict(X_test_TFIDF_camel.toarray())
print(classification_report(y_test_TFIDF_camel, y_pred_NB_TFIDF))

***Decision Tree - camel - TFIDF***

In [ ]:
DT_TFIDF = DecisionTreeClassifier()
DT_TFIDF.fit(X_train_TFIDF_camel.toarray(), y_train_TFIDF_camel)
y_pred_DT_TFIDF = DT_TFIDF.predict(X_test_TFIDF_camel.toarray())
print(classification_report(y_test_TFIDF_camel, y_pred_DT_TFIDF))

***Logistic Regression - TFIDF***

In [ ]:
LR_TFIDF= LogisticRegression()
LR_TFIDF.fit(X_train_TFIDF_camel.toarray(), y_train_TFIDF_camel)
y_pred_LR_TFIDF = LR_TFIDF.predict(X_test_TFIDF_camel.toarray())
print(classification_report(y_test_TFIDF_camel, y_pred_LR_TFIDF))